Contoh Soal Bayesian Optimization

In [32]:
import numpy as np
import pandas as pd
from bayes_opt import BayesianOptimization
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

# Memasukkan keseluruhan data olahan
data_olahan = {'Temperatur': [80, 83, 68, 64, 69, 71, 78, 82, 73, 77],
               'Kelembapan': [90, 78, 80, 65, 70, 80, 75, 92, 88, 70],
               'Jumlah_Pemain': [39, 43, 28, 43, 56, 13, 51, 41, 29, 36]}
df = pd.DataFrame(data_olahan)

# Penetapan variabel input dan variabel output
X, y = df.drop('Jumlah_Pemain', axis=1), df['Jumlah_Pemain']

# Splitting data olahan menjadi data latih dan data uji dengan perbandingan proporsi data 50%:50%
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=False)


In [33]:
# Menentukan hyperparameter target yang akan dioptimalkan beserta batasan nilainya
pbounds = {'min_samples_split': (1, 10),
           'max_depth': (2, 15)}

# Mendefinisikan fungsi representasi target hyperparameter yang akan dioptimalkan oleh algoritma Bayesian Optimization
def dt_cv_score(min_samples_split, max_depth):
    try:
        min_samples_split = int(min_samples_split)
        max_depth = int(max_depth)

        model = DecisionTreeRegressor(min_samples_split=min_samples_split,
                                      max_depth=max_depth,                                     
                                      random_state=42)
        
        tscv = TimeSeriesSplit(n_splits=3)

        scores = cross_val_score(model,
                                 X_train,
                                 y_train,
                                 cv=tscv,
                                 scoring='neg_root_mean_squared_error')

        return np.mean(scores)

    except Exception as e:
        print(f"Error dengan parameter: {e}")
        return -1e9 

# Menjalankan algoritma Bayesian Optimization untuk mengoptimalkan nilai hyperparameter target
optimizer = BayesianOptimization(f=dt_cv_score,  # Fungsi yang akan dioptimalkan
                                 pbounds=pbounds,  # Batasan parameter
                                 random_state=42)

optimizer.maximize(
    init_points=5,  
    n_iter=50) 

best_params = optimizer.max['params']

best_params_formatted = {'min_samples_split': int(best_params['min_samples_split']),
                         'max_depth': int(best_params['max_depth'])}

# Melihat nilai hyperparameter optimal hasil algoritma Bayesian Optimization
print(f'Hyperparameter terbaik: {best_params_formatted}')

|   iter    |  target   | min_sa... | max_depth |
-------------------------------------------------
| 1         | -10.77777 | 4.3708610 | 14.359285 |
| 2         | -12.36111 | 7.5879454 | 9.7825602 |
| 3         | -13.0     | 2.4041677 | 4.0279287 |
Error dengan parameter: 
All the 3 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\asus5\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\asus5\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
  File "c:\Users\asus5\AppData\Lo

In [34]:
# Melatih model Random Forest dengan nilai hyperparameter target sesuai hasil kinerja algoritma Bayesian Optimization
best_dt_model = DecisionTreeRegressor(**best_params_formatted,
                                      random_state=42)
best_dt_model.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=14, min_samples_split=4, random_state=42)

In [35]:
# Model Random Forest dengan Bayesian Optimization Hyperparameter Tuning melakukan prediksi terhadap variabel target 'Close_Diff'
y_pred = best_dt_model.predict(X_test)

# Menampilkan hasil prediksi
print(y_pred)

# Evaluasi model RF dengan metrik RMSE
rmse = root_mean_squared_error(y_test, y_pred)
print(f'RMSE: {rmse:.4f}')

[33.5        47.33333333 33.5        33.5        47.33333333]
RMSE: 11.3017
